# Демонстрационный notebook: преобразование, нормализация и стандартизация данных

Этот notebook сопровождает вебинар по подготовке аналитического датасета. Логика работы: загрузить исходные файлы, проверить структуру, привести типы, стандартизировать текстовые категории, создать признаки, объединить справочники, построить агрегаты, сводные таблицы и масштабировать числовые признаки.

**Сквозной кейс:** подготовка данных о продажах к аналитическому отчёту.


## 1. Импорт библиотек и настройка проекта

В этом блоке создаём единые пути к папкам. Notebook можно запускать как из корня проекта, так и из папки `notebooks`.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

def project_path(path):
    """Показывает путь относительно корня проекта, чтобы notebook был переносимым."""
    return Path(path).relative_to(PROJECT_ROOT)

print("Корень проекта: текущая папка проекта")
print("Папка исходных данных:", project_path(RAW_DIR))
print("Папка результатов:", project_path(OUTPUTS_DIR))


## 2. Проверка наличия файлов

Перед анализом проверяем, что все учебные файлы действительно лежат в `data/raw`. Это снижает число технических ошибок на вебинаре.


In [ ]:
required_files = [
    RAW_DIR / "sales.csv",
    RAW_DIR / "products.xlsx",
    RAW_DIR / "regions.json",
    RAW_DIR / "clients.csv",
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    print("Не найдены файлы:")
    for path in missing_files:
        print("-", project_path(path))
    raise FileNotFoundError("Не все исходные файлы найдены. Проверьте структуру проекта.")
else:
    print("Все исходные файлы найдены.")


## 3. Загрузка данных

Загружаем четыре источника: продажи, товары, регионы и клиентов.


In [ ]:
sales = pd.read_csv(RAW_DIR / "sales.csv")
products = pd.read_excel(RAW_DIR / "products.xlsx", sheet_name="products")
regions = pd.DataFrame(json.loads((RAW_DIR / "regions.json").read_text(encoding="utf-8")))
clients = pd.read_csv(RAW_DIR / "clients.csv")

print("sales:", sales.shape)
print("products:", products.shape)
print("regions:", regions.shape)
print("clients:", clients.shape)


## 4. Первичный обзор таблицы продаж

Смотрим первые строки, типы данных и базовую информацию. `head()` показывает внешний вид, а `dtypes` показывает, как pandas понял столбцы.


In [ ]:
display(sales.head())
display(sales.dtypes.to_frame("dtype"))


## 5. Преобразование даты

Дата заказа должна быть датой, а не обычной строкой. Некорректные даты переводим в `NaT`, чтобы найти и обработать их явно.


In [ ]:
sales_work = sales.copy()

sales_work["order_date_raw"] = sales_work["order_date"]
sales_work["order_date"] = pd.to_datetime(
    sales_work["order_date"],
    errors="coerce",
    format="mixed",
    dayfirst=True,
)

invalid_dates = sales_work[sales_work["order_date"].isna()][["order_id", "order_date_raw"]]
print("Строк с нераспознанной датой:", len(invalid_dates))
display(invalid_dates.head())


## 6. Календарные признаки

Из даты создаём месяц, квартал и день недели. Эти признаки понадобятся для динамики и отчётных разрезов.


In [ ]:
sales_work["order_month"] = sales_work["order_date"].dt.to_period("M").astype("string")
sales_work["order_quarter"] = sales_work["order_date"].dt.to_period("Q").astype("string")
sales_work["order_weekday"] = sales_work["order_date"].dt.day_name()

# Для дальнейших расчётов исключаем строки с нераспознанной датой.
sales_work = sales_work[sales_work["order_date"].notna()].copy()

display(sales_work[["order_id", "order_date", "order_month", "order_quarter", "order_weekday"]].head())


## 7. Стандартизация текстовых категорий

Канал продаж должен быть записан единообразно. Для человека `Online`, ` online ` и `web` могут означать одно и то же, но для программы это разные значения.


In [ ]:
print("До стандартизации:")
display(sales_work["channel"].value_counts(dropna=False))

sales_work["channel"] = sales_work["channel"].str.strip().str.lower()

channel_map = {
    "web": "online",
    "internet": "online",
    "offline": "offline",
    "store": "offline",
    "partner": "partner",
}

sales_work["channel"] = sales_work["channel"].replace(channel_map)

print("После стандартизации:")
display(sales_work["channel"].value_counts(dropna=False))


## 8. Проверка числовых полей

Перед расчётами убеждаемся, что количество, цена и скидка являются числовыми полями.


In [ ]:
numeric_columns = ["quantity", "unit_price", "discount"]

for column in numeric_columns:
    sales_work[column] = pd.to_numeric(sales_work[column], errors="coerce")

check_numeric = sales_work[numeric_columns].isna().sum().to_frame("missing_after_numeric_conversion")
display(check_numeric)
display(sales_work[numeric_columns].describe())


## 9. Расчёт выручки

Выручку считаем из количества, цены и скидки. После создания нового признака обязательно проверяем результат.


In [ ]:
sales_work["revenue"] = sales_work["quantity"] * sales_work["unit_price"] * (1 - sales_work["discount"])

display(sales_work[["quantity", "unit_price", "discount", "revenue"]].head(10))

print("Минимальная выручка:", round(sales_work["revenue"].min(), 2))
print("Максимальная выручка:", round(sales_work["revenue"].max(), 2))
print("Пропуски в revenue:", sales_work["revenue"].isna().sum())


## 10. Проверка ключей перед объединением

Перед `merge` проверяем уникальность ключей в справочниках. Если в справочнике есть дубли по ключу, строки после объединения могут размножиться.


In [ ]:
reference_checks = pd.DataFrame({
    "table": ["products", "regions", "clients"],
    "rows": [len(products), len(regions), len(clients)],
    "unique_key_count": [
        products["product_id"].nunique(),
        regions["region_id"].nunique(),
        clients["client_id"].nunique(),
    ],
})
reference_checks["duplicate_keys"] = reference_checks["rows"] - reference_checks["unique_key_count"]
display(reference_checks)


## 11. Объединение продаж со справочниками

Используем `left merge`, чтобы сохранить все строки продаж и увидеть, где справочник не нашёл соответствие.


In [ ]:
prepared = sales_work.merge(products, on="product_id", how="left")
prepared = prepared.merge(regions, on="region_id", how="left")
prepared = prepared.merge(clients, on="client_id", how="left")

print("Строк до merge:", len(sales_work))
print("Строк после merge:", len(prepared))

merge_quality = pd.DataFrame({
    "field_from_reference": ["product_name", "region_name", "client_name"],
    "missing_count": [
        prepared["product_name"].isna().sum(),
        prepared["region_name"].isna().sum(),
        prepared["client_name"].isna().sum(),
    ]
})
display(merge_quality)


## 12. Диагностика несовпавших ключей

Если после объединения появились пропуски в полях справочника, нужно посмотреть, какие ключи не сопоставились.


In [ ]:
missing_products = prepared.loc[prepared["product_name"].isna(), "product_id"].drop_duplicates()
missing_clients = prepared.loc[prepared["client_name"].isna(), "client_id"].drop_duplicates()

print("Несовпавшие product_id:")
display(missing_products)

print("Несовпавшие client_id:")
display(missing_clients)


## 13. Заполнение понятных значений для учебного датасета

Для отчётной таблицы не оставляем непонятные пустые подписи. Отдельно фиксируем, что часть справочников не нашлась.


In [ ]:
prepared["product_found"] = prepared["product_name"].notna()
prepared["client_found"] = prepared["client_name"].notna()

prepared["product_name"] = prepared["product_name"].fillna("Неизвестный товар")
prepared["category"] = prepared["category"].fillna("unknown")
prepared["product_line"] = prepared["product_line"].fillna("unknown")
prepared["unit_cost"] = prepared["unit_cost"].fillna(0)

prepared["client_name"] = prepared["client_name"].fillna("Неизвестный клиент")
prepared["segment"] = prepared["segment"].fillna("unknown")
prepared["loyalty_level"] = prepared["loyalty_level"].fillna("unknown")

display(prepared.head())


## 14. Расчёт маржи

После объединения с товарами появился `unit_cost`, значит можно рассчитать валовую маржу.


In [ ]:
prepared["gross_margin"] = prepared["revenue"] - prepared["quantity"] * prepared["unit_cost"]
prepared["margin_rate"] = np.where(
    prepared["revenue"] != 0,
    prepared["gross_margin"] / prepared["revenue"],
    np.nan,
)

display(prepared[["revenue", "quantity", "unit_cost", "gross_margin", "margin_rate"]].head(10))


## 15. Агрегация по категориям

Теперь переходим от детальных строк продаж к управленческим показателям.


In [ ]:
category_summary = (
    prepared
    .groupby("category", as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean"),
        total_margin=("gross_margin", "sum"),
    )
    .sort_values("total_revenue", ascending=False)
)

display(category_summary)


## 16. Контроль суммы после группировки

Если мы не фильтровали строки, сумма выручки после группировки должна совпадать с исходной суммой.


In [ ]:
source_revenue = prepared["revenue"].sum()
grouped_revenue = category_summary["total_revenue"].sum()

print("Выручка в подготовленной таблице:", round(source_revenue, 2))
print("Выручка после группировки:", round(grouped_revenue, 2))
print("Разница:", round(source_revenue - grouped_revenue, 6))


## 17. Сводная таблица по категориям и каналам

`pivot_table` удобно использовать для отчётного вида: строки, столбцы и показатель внутри ячеек.


In [ ]:
category_by_channel = pd.pivot_table(
    prepared,
    index="category",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
)

display(category_by_channel)


## 18. Ещё один отчётный разрез: регионы и месяцы

Такой срез помогает увидеть динамику продаж по регионам.


In [ ]:
region_month = pd.pivot_table(
    prepared,
    index="region_name",
    columns="order_month",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
)

display(region_month)


## 19. Масштабирование числовых признаков

Масштабируем только осмысленные числовые признаки. Идентификаторы не масштабируем.


In [ ]:
features_for_scaling = prepared[["quantity", "unit_price", "discount", "revenue", "gross_margin"]].copy()

display(features_for_scaling.describe())


## 20. MinMaxScaler, StandardScaler, RobustScaler

Сравниваем три подхода к масштабированию. Для реальной задачи scaler выбирается по смыслу признаков и распределению данных.


In [ ]:
minmax_scaler = MinMaxScaler()
standard_scaler = StandardScaler()
robust_scaler = RobustScaler()

scaled_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(features_for_scaling),
    columns=[f"{column}_minmax" for column in features_for_scaling.columns],
    index=prepared.index,
)

scaled_standard = pd.DataFrame(
    standard_scaler.fit_transform(features_for_scaling),
    columns=[f"{column}_standard" for column in features_for_scaling.columns],
    index=prepared.index,
)

scaled_robust = pd.DataFrame(
    robust_scaler.fit_transform(features_for_scaling),
    columns=[f"{column}_robust" for column in features_for_scaling.columns],
    index=prepared.index,
)

display(scaled_minmax.head())
display(scaled_standard.head())
display(scaled_robust.head())


## 21. Сбор итогового датасета

Для отчёта сохраняем подготовленную таблицу и агрегаты. Масштабированные признаки сохраняем отдельно, чтобы не перегружать основную отчётную таблицу.


In [ ]:
prepared_for_export = prepared.copy()

prepared_path = PROCESSED_DIR / "sales_prepared.csv"
category_summary_path = OUTPUTS_DIR / "category_summary.csv"
category_by_channel_path = OUTPUTS_DIR / "category_by_channel.csv"
region_month_path = OUTPUTS_DIR / "region_month.csv"
scaled_features_path = OUTPUTS_DIR / "scaled_features.csv"

prepared_for_export.to_csv(prepared_path, index=False, encoding="utf-8-sig")
category_summary.to_csv(category_summary_path, index=False, encoding="utf-8-sig")
category_by_channel.to_csv(category_by_channel_path, encoding="utf-8-sig")
region_month.to_csv(region_month_path, encoding="utf-8-sig")

scaled_features = pd.concat([scaled_minmax, scaled_standard, scaled_robust], axis=1)
scaled_features.to_csv(scaled_features_path, index=False, encoding="utf-8-sig")

print("Сохранено:")
for path in [prepared_path, category_summary_path, category_by_channel_path, region_month_path, scaled_features_path]:
    print("-", project_path(path))


## 22. Финальный чек-лист

Проверьте себя:

- исходные файлы загружены;
- даты приведены к datetime;
- текстовые категории стандартизированы;
- выручка и маржа рассчитаны;
- справочники объединены через `left merge`;
- после `merge` проверено количество строк;
- несовпавшие ключи найдены и зафиксированы;
- агрегаты построены и проверены контрольной суммой;
- сводные таблицы отвечают на аналитический вопрос;
- масштабированы только осмысленные числовые признаки;
- итоговые файлы сохранены.
